In [1]:

import pandas as pd

from etl.config import get_staging_engine, get_source_engine, get_dwh_engine

# create engine
source_engine = get_source_engine()
staging_engine = get_staging_engine()
dwh_engine = get_dwh_engine()

# source query
source_customer_query = """
                        SELECT CustomerID
                             , CustomerCategoryID
                             , BuyingGroupID
                             , PrimaryContactPersonID
                             , DeliveryMethodID
                             , DeliveryCityID
                             , PostalCityID
                             , AccountOpenedDate
                             , StandardDiscountPercentage
                        FROM customers;
                        """

source_category_query = """
                        SELECT CustomerCategoryID
                             , CustomerCategoryName
                        FROM customercategories;
                        """

source_buyinggroups_query = """
                            SELECT BuyingGroupID,
                                   BuyingGroupName
                            FROM buyinggroups;
                            """

# save to staging
source_customer_df = pd.read_sql(source_customer_query, source_engine)
source_customer_df.to_sql('customers', con=staging_engine, index=False, if_exists='replace')

source_category_df = pd.read_sql(source_category_query, source_engine)
source_category_df.to_sql('customercategories', con=staging_engine, index=False, if_exists='replace')

source_buyinggroups_df = pd.read_sql(source_buyinggroups_query, source_engine)
source_buyinggroups_df.to_sql('buyinggroups', con=staging_engine, index=False, if_exists='replace')

staging_query = """
                SELECT c.CustomerID
                     , c.CustomerCategoryID
                     , cc.CustomerCategoryName
                     , c.BuyingGroupID
                     , b.BuyingGroupName
                     , c.PrimaryContactPersonID
                     , c.DeliveryMethodID
                     , c.DeliveryCityID
                     , c.PostalCityID
                     , c.AccountOpenedDate
                     , c.StandardDiscountPercentage
                FROM customers AS c
                         JOIN customercategories AS cc ON c.CustomerCategoryID = cc.CustomerCategoryID
                         LEFT JOIN buyinggroups AS b ON b.BuyingGroupID = c.BuyingGroupID;
                """


In [7]:

staging_df = pd.read_sql(staging_query, staging_engine)
staging_df['BuyingGroupID'] = staging_df['BuyingGroupID'].fillna(-1)
staging_df['BuyingGroupName'] = staging_df['BuyingGroupName'].fillna('Unknown')

staging_df

,CustomerID,CustomerCategoryID,CustomerCategoryName,BuyingGroupID,BuyingGroupName,PrimaryContactPersonID,DeliveryMethodID,DeliveryCityID,PostalCityID,AccountOpenedDate,StandardDiscountPercentage
0,1.0,3.0,Novelty Shop,1.0,Tailspin Toys,1001.0,3.0,19586.0,19586.0,2013-01-01,0.0
1,2.0,3.0,Novelty Shop,1.0,Tailspin Toys,1003.0,3.0,33475.0,33475.0,2013-01-01,0.0
2,3.0,3.0,Novelty Shop,1.0,Tailspin Toys,1005.0,3.0,26483.0,26483.0,2013-01-01,0.0
3,4.0,3.0,Novelty Shop,1.0,Tailspin Toys,1007.0,3.0,21692.0,21692.0,2013-01-01,0.0
4,5.0,3.0,Novelty Shop,1.0,Tailspin Toys,1009.0,3.0,12748.0,12748.0,2013-01-01,0.0
...,...,...,...,...,...,...,...,...,...,...,...
658,1057.0,5.0,Computer Store,-1.0,Unknown,3257.0,3.0,25608.0,25608.0,2016-01-09,0.0
659,1058.0,6.0,Gift Store,-1.0,Unknown,3258.0,3.0,31564.0,31564.0,2016-02-01,0.0
660,1059.0,4.0,Supermarket,-1.0,Unknown,3259.0,3.0,10483.0,10483.0,2016-03-13,0.0
661,1060.0,7.0,Corporate,-1.0,Unknown,3260.0,3.0,22090.0,22090.0,2016-04-23,0.0
